# 실험 010 — 동일 seed의 B0 4증강 앙상블

000에서 학습한 가중치를 재사용합니다. 추가 학습은 없습니다.
분할 seed 5, 네 모델의 학습 seed 5, basic·strong·strong+원근·strong+가림 구성입니다.
000의 같은 seed 4증강 검증 손실이 가장 낮았던 seed 5를 사용합니다.
프로젝트 루트에서 별도 커널로 전체 실행합니다. 기존 model.py·predict.py의 B0 4모델 구조와 추론 코드를 재사용하고 010 패키지에 복사합니다.
기본 T0.3·전체 T0.5·T0.3 TTA 제출 CSV를 함께 생성하고, TTA의 CPU 재현을 확인합니다.
T0.3은 전원 일치 시 T=0.3, 불일치 시 T=0.7입니다. 전원 일치이면 네 확률 평균, 불일치이면 가운데 두 확률 평균을 사용합니다.
실제 Colab CPU 시간은 별도 확인 대상입니다.


In [ ]:
import csv
import hashlib
import io
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from model import IMG_SIZE, NORM_MEAN, NORM_STD, EfficientNetEnsemble, build_valid_transform, combine_probabilities

BASE_DIR = Path.cwd().resolve()
SEED = 5
SPLIT_SEED = 5
AUGMENTATIONS = ("basic", "strong", "perspective", "erasing")
SOURCE_DIR = BASE_DIR / "실험결과" / "000_모델증강비교" / f"split_{SPLIT_SEED}"
RESULT_DIR = BASE_DIR / "실험결과" / "010_동일시드4증강앙상블" / f"seed_{SEED}"
PACKAGE_DIR = RESULT_DIR / "package"
assert SEED in (5, 6, 7), "000에서 학습 완료된 seed를 사용하세요."
assert (BASE_DIR / "baseline_test4.ipynb").is_file(), "프로젝트 루트에서 실행하세요."
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)
(RESULT_DIR / "코드").mkdir(exist_ok=True)
torch.set_num_threads(4)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cuda.matmul.allow_tf32 = False
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


def file_hash(path):
	return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def read_csv(path):
	with Path(path).open(newline="", encoding="utf-8-sig") as f:
		return list(csv.DictReader(f))


def write_csv(path, rows):
	assert rows, "저장할 결과가 없습니다."
	buffer = io.StringIO(newline="")
	writer = csv.DictWriter(buffer, fieldnames=list(rows[0]), lineterminator="\n")
	writer.writeheader()
	writer.writerows(rows)
	body = buffer.getvalue()
	if path.exists():
		assert path.read_text(encoding="utf-8-sig") == body, f"기존 CSV와 결과가 다릅니다: {path}"
	else:
		with path.open("w", newline="", encoding="utf-8-sig") as f:
			f.write(body)


print("010 구성:", [(augmentation, SEED) for augmentation in AUGMENTATIONS], "/ 추가 학습 없음", flush=True)


In [ ]:
validation_rows = read_csv(SOURCE_DIR / "validation_split.csv")
valid_ids = [row["id"] for row in validation_rows]
valid_labels = np.array([int(row["label"]) for row in validation_rows])
assert len(valid_ids) == len(set(valid_ids)) == 436
state_dicts, source_records, saved_probabilities = [], [], []
for augmentation in AUGMENTATIONS:
	run_id = f"efficientnet_b0_{augmentation}_seed{SEED}"
	directory = SOURCE_DIR / "runs" / run_id
	completion = json.loads((directory / "completed.json").read_text(encoding="utf-8"))
	config = completion["config"]
	assert (config["model"] == "efficientnet_b0") and (config["augmentation"] == augmentation) and (config["seed"] == SEED) and (config["split_seed"] == SPLIT_SEED)
	assert (config["image_size"] == IMG_SIZE) and (config["normalization_mean"] == list(NORM_MEAN)) and (config["normalization_std"] == list(NORM_STD)) and (config["tf32"] is False)
	for filename in ("checkpoint.pt", "validation_predictions.csv"):
		assert file_hash(directory / filename) == completion["artifact_sha256"][filename], f"원본 산출물이 변경됐습니다: {run_id}/{filename}"
	rows = read_csv(directory / "validation_predictions.csv")
	assert [row["id"] for row in rows] == valid_ids
	assert [int(row["label"]) for row in rows] == valid_labels.tolist()
	saved_probabilities.append([float(row["prob"]) for row in rows])
	state_dicts.append(torch.load(directory / "checkpoint.pt", map_location="cpu", weights_only=True))
	source_records.append({"run_id": run_id, "checkpoint": str((directory / "checkpoint.pt").relative_to(BASE_DIR)), "checkpoint_sha256": completion["artifact_sha256"]["checkpoint.pt"], "config": config})
for record in source_records[1:]:
	for key in ("dataset_sha256", "training_source_sha256", "params"):
		assert record["config"][key] == source_records[0]["config"][key], f"재사용 모델의 공통 학습 조건이 다릅니다: {key}"
training_hash = source_records[0]["config"]["training_source_sha256"]
source_notebook = SOURCE_DIR / "코드" / f"model_transform_test_{training_hash[:12]}.ipynb"
training_notebook = json.loads(source_notebook.read_text(encoding="utf-8"))
training_source = "".join("".join(cell["source"]) for cell in training_notebook["cells"] if cell["id"] in {"data-models", "training"})
assert hashlib.sha256(training_source.encode()).hexdigest() == training_hash
model = EfficientNetEnsemble(state_dicts).to(device).eval()
n_params = sum(parameter.numel() for parameter in model.parameters())
assert n_params == 16_035_316
bundle = {"state_dicts": state_dicts, "seeds": [SEED] * 4, "split_seed": SPLIT_SEED, "augmentations": list(AUGMENTATIONS), "image_size": IMG_SIZE, "preprocessing": "aspect_resize_pad", "temperature_rules": ["T0.3", "T0.5"], "temperatures": [0.3, 0.7], "source_records": source_records}
ensemble_path = RESULT_DIR / "ensemble.pt"
if ensemble_path.exists():
	existing = torch.load(ensemble_path, map_location="cpu", weights_only=True)
	assert {key: value for key, value in existing.items() if key != "state_dicts"} == {key: value for key, value in bundle.items() if key != "state_dicts"}, "기존 010 가중치의 구성이 다릅니다."
	for old_state, new_state in zip(existing["state_dicts"], state_dicts):
		assert old_state.keys() == new_state.keys()
		assert all(torch.equal(old_state[key], new_state[key]) for key in new_state), "기존 010 가중치와 값이 다릅니다."
else:
	torch.save(bundle, ensemble_path)
del state_dicts, bundle
print("000 가중치 4개 확인 완료 / 장치:", device, "/ 파라미터:", f"{n_params:,}", flush=True)


In [ ]:
transform = build_valid_transform()


def predict_members(paths):
	base_batches, tta_batches = [], []
	with torch.inference_mode():
		for start in range(0, len(paths), 32):
			images = []
			for path in paths[start:start + 32]:
				with Image.open(path) as image:
					images.append(transform(image.convert("RGB")))
			inputs = torch.stack(images).to(device)
			base = model.member_probabilities(inputs)
			flipped = model.member_probabilities(inputs.flip(dims=[3]))
			base_batches.append(base.cpu().numpy())
			tta_batches.append(((base + flipped) / 2).cpu().numpy())
	return np.concatenate(base_batches).astype(np.float64), np.concatenate(tta_batches).astype(np.float64)


def ensemble_predictions(base, tta):
	assert np.isfinite(base).all() and np.isfinite(tta).all()
	assert ((base >= 0) & (base <= 1)).all() and ((tta >= 0) & (tta <= 1)).all()
	return {
		"T0.3": combine_probabilities(torch.from_numpy(base), "T0.3").numpy().reshape(-1),
		"T0.5": combine_probabilities(torch.from_numpy(base), "T0.5").numpy().reshape(-1),
		"T0.3_TTA": combine_probabilities(torch.from_numpy(tta), "T0.3").numpy().reshape(-1),
	}


valid_paths = [BASE_DIR / "train" / ("it" if label == 1 else "6e") / (image_id + ".jpg") for image_id, label in zip(valid_ids, valid_labels)]
valid_base, valid_tta = predict_members(valid_paths)
reproduction_diff = float(np.abs(valid_base - np.array(saved_probabilities).T).max())
assert reproduction_diff <= 0.00001, f"000의 개별 검증 예측과 다릅니다: {reproduction_diff}"
valid_predictions = ensemble_predictions(valid_base, valid_tta)
metric_rows = []
for name, probabilities in [(augmentation, valid_base[:, index]) for index, augmentation in enumerate(AUGMENTATIONS)] + list(valid_predictions.items()):
	true_probs = np.where(valid_labels == 1, probabilities, 1 - probabilities)
	errors = int(((probabilities >= 0.5) != valid_labels.astype(bool)).sum())
	loss = float(-np.log(np.clip(true_probs, 1e-6, 1 - 1e-6)).mean())
	metric_rows.append({"model": name, "samples": len(valid_ids), "errors": errors, "valid_acc": 1 - errors / len(valid_ids), "valid_loss": loss})
	print("검증:", name, "/ 오답:", errors, "/ Log Loss:", f"{loss:.8f}", flush=True)
write_csv(RESULT_DIR / "ensemble_metrics.csv", metric_rows)
validation_predictions = []
for index, image_id in enumerate(valid_ids):
	row = {"id": image_id, "label": int(valid_labels[index])}
	row.update({f"{augmentation}_prob": float(valid_base[index, member_index]) for member_index, augmentation in enumerate(AUGMENTATIONS)})
	row.update({f"{augmentation}_TTA_prob": float(valid_tta[index, member_index]) for member_index, augmentation in enumerate(AUGMENTATIONS)})
	row.update({f"{rule}_prob": float(probabilities[index]) for rule, probabilities in valid_predictions.items()})
	validation_predictions.append(row)
write_csv(RESULT_DIR / "validation_predictions.csv", validation_predictions)
print("000 개별 검증 확률과 최대 차이:", reproduction_diff, flush=True)

test_paths = sorted((BASE_DIR / "test").glob("*.jpg"))
test_ids = [path.stem for path in test_paths]
expected_ids = [row["id"] for row in read_csv(BASE_DIR / "sample_submission.csv")]
assert len(test_ids) == len(set(test_ids)) == len(expected_ids) == 500
assert set(test_ids) == set(expected_ids)
test_base, test_tta = predict_members(test_paths)
test_predictions = ensemble_predictions(test_base, test_tta)
for rule, probabilities in test_predictions.items():
	assert np.isfinite(probabilities).all() and ((probabilities >= 0) & (probabilities <= 1)).all()
	path = RESULT_DIR / f"submission_{SEED}_{rule}.csv"
	write_csv(path, [{"id": image_id, "prob": f"{probability:.6f}"} for image_id, probability in zip(test_ids, probabilities)])
	print("제출 CSV:", path.resolve(), flush=True)
del model
if device.type == "cuda":
	torch.cuda.empty_cache()


In [ ]:
for filename in ("model.py", "predict.py", "baseline_test4.ipynb"):
	shutil.copy2(BASE_DIR / filename, PACKAGE_DIR / filename)
shutil.copy2(BASE_DIR / "baseline_test4.ipynb", RESULT_DIR / "코드" / "baseline_test4.ipynb")
shutil.copy2(ensemble_path, PACKAGE_DIR / "model.pt")
assert file_hash(PACKAGE_DIR / "model.pt") == file_hash(ensemble_path)
(PACKAGE_DIR / "source_000").mkdir(exist_ok=True)
shutil.copy2(source_notebook, PACKAGE_DIR / "source_000" / source_notebook.name)
inference_source = {filename: file_hash(PACKAGE_DIR / filename) for filename in ("model.py", "predict.py", "baseline_test4.ipynb")}
source_report = {"experiment": "010", "split_seed": SPLIT_SEED, "model_seeds": [SEED] * 4, "augmentations": list(AUGMENTATIONS), "members": source_records, "parameters": n_params, "tf32": False, "tta": "각 모델의 원본·좌우 반전 확률 평균 후 결합", "temperature": {"unanimous": 0.3, "disagreement": 0.7}, "validation": metric_rows, "validation_reproduction_max_difference": reproduction_diff, "inference_source_sha256": inference_source, "submission_sha256": {rule: file_hash(RESULT_DIR / f"submission_{SEED}_{rule}.csv") for rule in test_predictions}}
(RESULT_DIR / "source_records.json").write_text(json.dumps(source_report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
shutil.copy2(RESULT_DIR / "source_records.json", PACKAGE_DIR / "source_records.json")

print("패키지 CPU TTA 500장 재현 확인 중", flush=True)
check_path = RESULT_DIR / f"package_check_{SEED}_T0.3_TTA.csv"
environment = dict(os.environ, OMP_NUM_THREADS="4", MKL_NUM_THREADS="4", PYTHONIOENCODING="utf-8")
started = time.perf_counter()
completed = subprocess.run([sys.executable, str(PACKAGE_DIR / "predict.py"), "--input", str(BASE_DIR / "test"), "--output", str(check_path), "--tta"], check=True, capture_output=True, text=True, encoding="utf-8", env=environment)
cpu_seconds = time.perf_counter() - started
print(completed.stdout.strip(), flush=True)
submitted = {row["id"]: float(row["prob"]) for row in read_csv(RESULT_DIR / f"submission_{SEED}_T0.3_TTA.csv")}
reproduced = {row["id"]: float(row["prob"]) for row in read_csv(check_path)}
assert len(submitted) == len(reproduced) == 500
assert submitted.keys() == reproduced.keys()
cpu_difference = max(abs(submitted[image_id] - reproduced[image_id]) for image_id in submitted)
assert cpu_difference <= 0.00001, f"CPU 재현 차이가 큽니다: {cpu_difference}"
assert cpu_seconds <= 600, "로컬 CPU 추론이 600초를 초과했습니다."
check_report = {"local_cpu_seconds": cpu_seconds, "tta_csv_max_difference": cpu_difference, "colab_cpu_verified": False}
(RESULT_DIR / "package_check.json").write_text(json.dumps(check_report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
zip_path = RESULT_DIR / "submission_package.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
	for path in sorted(PACKAGE_DIR.rglob("*")):
		if path.is_file() and ("__pycache__" not in path.parts):
			archive.write(path, path.relative_to(PACKAGE_DIR))
assert zip_path.stat().st_size <= 200_000_000
with zipfile.ZipFile(zip_path) as archive:
	assert archive.testzip() is None
print("010 완료 / CPU 초:", round(cpu_seconds, 2), "/ CSV 최대 차이:", cpu_difference, "/ ZIP 바이트:", zip_path.stat().st_size, flush=True)
